# Posit Assistant Agentic Resources: EZPill Clinical Trial

This notebook is my testbed for exploring Posit Assistant's agentic resources through a fictional clinical research workflow. I am acting as a clinical trial analyst at a contract research organization (CRO), working with a sponsor to explore data and analyses relevant to a hypothetical New Drug Application (NDA).

## Study Overview

| Study Feature | Fictional Trial Details |
| --- | --- |
| Study ID / Protocol | EZP-301 / EZP-301-PROT-01 |
| Development phase | Phase III training scenario: a late-stage study intended to evaluate efficacy and safety in a larger population |
| Investigational product | EZPill, a fictional oral cholesterol-lowering drug; no specific mechanism of action is modeled |
| Treatment arms | EZPill 20 mg once daily versus placebo once daily |
| Design | Simulated randomized, double-blind, placebo-controlled, parallel-group study |
| Duration | 52 weeks per participant |
| Population | 1,200 synthetic adults aged 40-80 years |
| Allocation | Approximately 1:1 treatment allocation within each baseline LDL-C stratum |
| Scheduled assessments | Baseline and Weeks 4, 12, 24, 36, and 52 |

## Population and Research Questions

**LDL-C** means low-density lipoprotein cholesterol, measured here in milligrams per deciliter (mg/dL). Elevated LDL-C is associated with increased cardiovascular risk.

The simulation assigns approximately 80% of participants to an elevated-LDL-C stratum (baseline LDL-C at least 130 mg/dL) and 20% to a normal-LDL-C stratum (below 130 mg/dL). Both strata include EZPill and placebo participants. These thresholds are simulation conventions, not universal clinical treatment criteria. The normal-LDL-C group is exploratory and is not a recommendation to treat people with normal cholesterol.

The main efficacy question for this exercise is whether LDL-C changes differently over 52 weeks in the EZPill and placebo groups, particularly among participants with elevated baseline LDL-C. Potential analyses include absolute and percentage change from baseline, longitudinal lipid profiles, adverse events, and study discontinuation. No formal primary endpoint, estimand, or statistical analysis plan has been implemented yet.

## Synthetic Data Resources

The Python code creates linked tables in `trial_data`: participant demographics and baseline characteristics (`subjects`), scheduled lipid and safety assessments (`labs`), adverse events (`adverse_events`), participant completion and discontinuation records (`disposition`), a partial variable dictionary (`data_dictionary`), and study-level metadata (`metadata`). Subjects are linked across tables by `subject_id`.

The simulation includes missed visits, missing laboratory values, discontinuations, and a programmed treatment effect to support analysis practice. Its fixed random seed makes the generated observations reproducible. Any apparent efficacy or safety findings reflect the simulation assumptions, not evidence about a real drug.

**Training use only:** EZPill, the protocol, participants, and observations are entirely fictional. These are simplified research tables, not validated CDISC SDTM/ADaM datasets or submission-ready NDA materials. The Phase III designation describes the educational scenario; this simulation cannot establish clinical benefit, safety, or regulatory suitability.

In [2]:
"""Generate a reproducible, fully synthetic EZPill trial dataset.

This is fictional training data, not clinical evidence or patient data. The
normal-LDL stratum is exploratory and would not represent the indicated
population for a cholesterol-lowering NDA.
"""

import numpy as np
import pandas as pd

SEED = 20261113
N_SUBJECTS = 1200
STUDY_ID = "EZP-301"
PROTOCOL_ID = "EZP-301-PROT-01"
RANDOMIZATION_DATE = pd.Timestamp("2025-01-06")
VISIT_WEEKS = [0, 4, 12, 24, 36, 52]

rng = np.random.default_rng(SEED)
subject_ids = [f"EZP301-{number:05d}" for number in range(1, N_SUBJECTS + 1)]
cholesterol_strata = rng.choice(
    ["Elevated LDL-C", "Normal LDL-C"],
    size=N_SUBJECTS,
    p=[0.80, 0.20],
)

# Balanced 1:1 allocation within each baseline LDL-C stratum.
treatment_arms = np.empty(N_SUBJECTS, dtype=object)
for stratum in np.unique(cholesterol_strata):
    stratum_indices = np.flatnonzero(cholesterol_strata == stratum)
    assignments = np.resize(["EZPill 20 mg QD", "Placebo QD"], len(stratum_indices))
    rng.shuffle(assignments)
    treatment_arms[stratum_indices] = assignments

age = rng.integers(40, 81, size=N_SUBJECTS)
sex = rng.choice(["Female", "Male"], size=N_SUBJECTS, p=[0.51, 0.49])
height_cm = np.clip(rng.normal(np.where(sex == "Male", 176, 163), 8), 145, 202)
bmi = np.clip(rng.normal(29.1, 5.2, size=N_SUBJECTS), 18.0, 47.0)
weight_kg = bmi * (height_cm / 100) ** 2
baseline_ldl = np.where(
    cholesterol_strata == "Elevated LDL-C",
    np.clip(rng.normal(158, 19, size=N_SUBJECTS), 130, 230),
    np.clip(rng.normal(108, 13, size=N_SUBJECTS), 65, 129.9),
)
baseline_hdl = np.clip(
    rng.normal(np.where(sex == "Male", 46, 55), 10, size=N_SUBJECTS), 25, 95
)
baseline_triglycerides = np.clip(rng.lognormal(np.log(137), 0.42, size=N_SUBJECTS), 45, 600)
baseline_total_cholesterol = baseline_ldl + baseline_hdl + baseline_triglycerides / 5

randomization_dates = RANDOMIZATION_DATE + pd.to_timedelta(
    rng.integers(0, 300, size=N_SUBJECTS), unit="D"
)
randomization_numbers = rng.permutation(np.arange(1, N_SUBJECTS + 1))
site_numbers = rng.integers(1, 61, size=N_SUBJECTS)

subjects = pd.DataFrame(
    {
        "study_id": STUDY_ID,
        "protocol_id": PROTOCOL_ID,
        "subject_id": subject_ids,
        "site_id": [f"SITE-{number:03d}" for number in site_numbers],
        "country": rng.choice(["USA", "Canada", "Germany", "Spain"], size=N_SUBJECTS, p=[0.55, 0.15, 0.18, 0.12]),
        "randomization_number": randomization_numbers,
        "randomization_date": randomization_dates,
        "treatment_arm": treatment_arms,
        "age_years": age,
        "sex": sex,
        "race": rng.choice(["White", "Black or African American", "Asian", "Other", "Not reported"], size=N_SUBJECTS, p=[0.62, 0.14, 0.12, 0.07, 0.05]),
        "ethnicity": rng.choice(["Hispanic or Latino", "Not Hispanic or Latino", "Not reported"], size=N_SUBJECTS, p=[0.18, 0.77, 0.05]),
        "height_cm": np.round(height_cm, 1),
        "weight_kg": np.round(weight_kg, 1),
        "bmi_kg_m2": np.round(bmi, 1),
        "smoking_status": rng.choice(["Never", "Former", "Current"], size=N_SUBJECTS, p=[0.55, 0.31, 0.14]),
        "diabetes_history": rng.choice(["Yes", "No"], size=N_SUBJECTS, p=[0.19, 0.81]),
        "background_statin": rng.choice(["Yes", "No"], size=N_SUBJECTS, p=[0.48, 0.52]),
        "baseline_ldl_stratum": cholesterol_strata,
        "baseline_total_cholesterol_mg_dl": np.round(baseline_total_cholesterol, 1),
        "baseline_ldl_c_mg_dl": np.round(baseline_ldl, 1),
        "baseline_hdl_c_mg_dl": np.round(baseline_hdl, 1),
        "baseline_triglycerides_mg_dl": np.round(baseline_triglycerides, 1),
    }
)

completion_probability = rng.choice([0.88, 0.93, 0.97], size=N_SUBJECTS, p=[0.10, 0.25, 0.65])
completed_study = rng.random(N_SUBJECTS) < completion_probability
disposition_reason = np.where(completed_study, "Completed", "")
discontinuation_week = np.full(N_SUBJECTS, 52)
for index in np.flatnonzero(~completed_study):
    discontinuation_week[index] = rng.choice([4, 12, 24, 36])
    disposition_reason[index] = rng.choice(
        ["Adverse event", "Withdrawal by subject", "Lost to follow-up", "Other"],
        p=[0.28, 0.34, 0.28, 0.10],
    )

visit_rows = []
for index, subject_id in enumerate(subject_ids):
    arm = treatment_arms[index]
    for week in VISIT_WEEKS:
        visit_done = week == 0 or (
            week <= discontinuation_week[index]
            and rng.random() < {4: 0.98, 12: 0.96, 24: 0.94, 36: 0.92, 52: 0.90}[week]
        )
        planned_date = randomization_dates[index] + pd.Timedelta(weeks=week)
        actual_date = (
            planned_date if week == 0 else planned_date + pd.Timedelta(days=int(rng.integers(-4, 5)))
        ) if visit_done else pd.NaT

        treatment_effect = 0.0
        if week > 0:
            treatment_effect = (0.22 if arm == "EZPill 20 mg QD" else 0.025) * (1 - np.exp(-week / 7))
        ldl_value = baseline_ldl[index] * (1 - treatment_effect) + rng.normal(0, 7)
        hdl_value = baseline_hdl[index] + (2.5 if arm == "EZPill 20 mg QD" else 0.5) * (week / 52) + rng.normal(0, 2.5)
        triglycerides_value = baseline_triglycerides[index] * (1 - 0.04 * week / 52) + rng.normal(0, 12)
        total_value = ldl_value + hdl_value + triglycerides_value / 5
        alt_value = np.clip(rng.normal(24, 10), 5, 120)
        ast_value = np.clip(rng.normal(23, 8), 5, 100)
        creatinine_value = np.clip(rng.normal(0.92, 0.18), 0.4, 2.5)
        systolic_bp = np.clip(rng.normal(128, 15), 85, 205)
        diastolic_bp = np.clip(rng.normal(78, 10), 45, 125)

        analytes = {
            "total_cholesterol_mg_dl": total_value,
            "ldl_c_mg_dl": ldl_value,
            "hdl_c_mg_dl": hdl_value,
            "triglycerides_mg_dl": triglycerides_value,
            "alt_u_l": alt_value,
            "ast_u_l": ast_value,
            "creatinine_mg_dl": creatinine_value,
        }
        if visit_done:
            for analyte in analytes:
                if rng.random() < 0.025:
                    analytes[analyte] = np.nan
        else:
            analytes = {analyte: np.nan for analyte in analytes}

        visit_rows.append(
            {
                "study_id": STUDY_ID,
                "subject_id": subject_id,
                "visit_name": "Baseline" if week == 0 else f"Week {week}",
                "visit_week": week,
                "visit_status": "Completed" if visit_done else "Not done",
                "planned_visit_date": planned_date,
                "actual_visit_date": actual_date,
                "total_cholesterol_mg_dl": analytes["total_cholesterol_mg_dl"],
                "ldl_c_mg_dl": analytes["ldl_c_mg_dl"],
                "hdl_c_mg_dl": analytes["hdl_c_mg_dl"],
                "triglycerides_mg_dl": analytes["triglycerides_mg_dl"],
                "alt_u_l": analytes["alt_u_l"],
                "ast_u_l": analytes["ast_u_l"],
                "creatinine_mg_dl": analytes["creatinine_mg_dl"],
                "systolic_bp_mmhg": round(systolic_bp, 1) if visit_done else np.nan,
                "diastolic_bp_mmhg": round(diastolic_bp, 1) if visit_done else np.nan,
            }
        )

labs = pd.DataFrame(visit_rows)

adverse_event_terms = [
    "Headache", "Nausea", "Diarrhea", "Myalgia", "Fatigue", "Dizziness", "Abdominal discomfort"
]
ae_counts = rng.poisson(0.55, size=N_SUBJECTS)
ae_rows = []
for index, event_count in enumerate(ae_counts):
    for event_number in range(1, event_count + 1):
        onset_day = int(rng.integers(1, 365))
        severity = rng.choice(["Mild", "Moderate", "Severe"], p=[0.70, 0.26, 0.04])
        serious = rng.random() < 0.025
        related_probability = 0.22 if treatment_arms[index] == "EZPill 20 mg QD" else 0.12
        ae_rows.append(
            {
                "study_id": STUDY_ID,
                "subject_id": subject_ids[index],
                "ae_sequence": event_number,
                "ae_term": rng.choice(adverse_event_terms),
                "onset_date": randomization_dates[index] + pd.Timedelta(days=onset_day),
                "duration_days": int(rng.integers(1, 22)),
                "severity": severity,
                "serious": "Yes" if serious else "No",
                "relationship_to_study_drug": "Related" if rng.random() < related_probability else "Not related",
                "action_taken": rng.choice(["None", "Dose interrupted", "Study drug discontinued"], p=[0.78, 0.16, 0.06]),
                "outcome": rng.choice(["Recovered", "Recovering", "Not recovered"], p=[0.86, 0.10, 0.04]),
            }
        )

adverse_events = pd.DataFrame(
    ae_rows,
    columns=[
        "study_id", "subject_id", "ae_sequence", "ae_term", "onset_date", "duration_days",
        "severity", "serious", "relationship_to_study_drug", "action_taken", "outcome"
    ],
)

disposition = pd.DataFrame(
    {
        "study_id": STUDY_ID,
        "subject_id": subject_ids,
        "end_of_study_status": np.where(completed_study, "Completed", "Discontinued"),
        "discontinuation_reason": disposition_reason,
        "last_visit_week": discontinuation_week,
        "end_of_study_date": [
            randomization_dates[index] + pd.Timedelta(weeks=int(discontinuation_week[index]))
            for index in range(N_SUBJECTS)
        ],
    }
)

trial_metadata = {
    "study_id": STUDY_ID,
    "protocol_id": PROTOCOL_ID,
    "product": "EZPill (fictional investigational product)",
    "design": "52-week randomized, double-blind, placebo-controlled, parallel-group simulation",
    "allocation": "1:1 within baseline LDL-C stratum",
    "population_note": "Normal-LDL-C stratum is exploratory; this simulated cohort is not an NDA population recommendation.",
    "random_seed": SEED,
    "synthetic_subject_count": N_SUBJECTS,
    "data_status": "Entirely synthetic; no real participants or clinical observations",
    "created_date": pd.Timestamp.today().normalize(),
}

data_dictionary = pd.DataFrame(
    [
        ("subjects", "subject_id", "Synthetic participant identifier", "Character", "None"),
        ("subjects", "site_id", "Synthetic investigative site identifier", "Character", "None"),
        ("subjects", "treatment_arm", "Randomized treatment assignment", "Categorical", "None"),
        ("subjects", "baseline_ldl_stratum", "Baseline LDL-C stratum used for randomization", "Categorical", "None"),
        ("subjects", "age_years", "Age at informed consent", "Integer", "years"),
        ("subjects", "bmi_kg_m2", "Body mass index", "Numeric", "kg/m^2"),
        ("subjects", "baseline_ldl_c_mg_dl", "Baseline low-density lipoprotein cholesterol", "Numeric", "mg/dL"),
        ("labs", "visit_week", "Nominal study week for the assessment", "Integer", "weeks"),
        ("labs", "visit_status", "Whether the scheduled visit was completed", "Categorical", "None"),
        ("labs", "ldl_c_mg_dl", "Low-density lipoprotein cholesterol", "Numeric", "mg/dL"),
        ("labs", "hdl_c_mg_dl", "High-density lipoprotein cholesterol", "Numeric", "mg/dL"),
        ("labs", "triglycerides_mg_dl", "Triglycerides", "Numeric", "mg/dL"),
        ("labs", "alt_u_l", "Alanine aminotransferase", "Numeric", "U/L"),
        ("labs", "ast_u_l", "Aspartate aminotransferase", "Numeric", "U/L"),
        ("adverse_events", "ae_term", "Fictional adverse event term; not a coded MedDRA term", "Categorical", "None"),
        ("adverse_events", "severity", "Simulated event severity", "Categorical", "None"),
        ("disposition", "end_of_study_status", "Participant completion status", "Categorical", "None"),
    ],
    columns=["table", "variable", "label", "data_type", "unit"],
)

trial_data = {
    "subjects": subjects,
    "labs": labs,
    "adverse_events": adverse_events,
    "disposition": disposition,
    "data_dictionary": data_dictionary,
    "metadata": trial_metadata,
}

dataset_overview = {
    "metadata": trial_metadata,
    "table_rows": {
        table_name: len(table)
        for table_name, table in trial_data.items()
        if isinstance(table, pd.DataFrame)
    },
    "subjects_by_treatment_and_stratum": subjects.groupby(
        ["treatment_arm", "baseline_ldl_stratum"], observed=True
    ).size(),
    "scheduled_lab_visits": len(labs),
    "adverse_event_records": len(adverse_events),
}
dataset_overview

{'metadata': {'study_id': 'EZP-301',
  'protocol_id': 'EZP-301-PROT-01',
  'product': 'EZPill (fictional investigational product)',
  'design': '52-week randomized, double-blind, placebo-controlled, parallel-group simulation',
  'allocation': '1:1 within baseline LDL-C stratum',
  'population_note': 'Normal-LDL-C stratum is exploratory; this simulated cohort is not an NDA population recommendation.',
  'random_seed': 20261113,
  'synthetic_subject_count': 1200,
  'data_status': 'Entirely synthetic; no real participants or clinical observations',
  'created_date': Timestamp('2026-11-13 00:00:00')},
 'table_rows': {'subjects': 1200,
  'labs': 7200,
  'adverse_events': 658,
  'disposition': 1200,
  'data_dictionary': 17},
 'subjects_by_treatment_and_stratum': treatment_arm    baseline_ldl_stratum
 EZPill 20 mg QD  Elevated LDL-C          479
                  Normal LDL-C            121
 Placebo QD       Elevated LDL-C          479
                  Normal LDL-C            121
 dtype: int